In [3]:
import pandas as pd
import numpy as np

# LOADED THE MESSY DATA
df = pd.read_csv('marketing_campaign_data_messy.csv')

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 2020 rows, 12 columns


In [4]:
# STEP 1: HEADER CLEANING

print(df.columns.tolist())
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("----- FIX APPLIED -----")
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
----- FIX APPLIED -----
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


In [5]:
# STEP 2: TYPE CONVERSION & CURRENCY CLEANING

dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask, ['campaign_id', 'spend']].head(3))

df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'])

print("----- FIX APPLIED -----")
print(df.loc[dirty_spend_mask, ['campaign_id', 'spend']].head(3)) # to compare

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
----- FIX APPLIED -----
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


In [6]:
# STEP 3: CATEGORICAL TYPOS (FUZZY LOGIC)

print(df['channel'].unique())

cleanup_map = {
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram',
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'N/A': np.nan # Handling the ghost value here too

}

df['channel'] = df['channel'].replace(cleanup_map)

print("----- FIX APPLIED -----")
print(df['channel'].unique())

<StringArray>
[    'TikTok',   'Facebook',      'Email',  'Instagram', 'Google Ads',
     'E-mail',          nan,      'Gogle',    'Tik_Tok',    'Facebok',
 'Insta_gram']
Length: 11, dtype: str
----- FIX APPLIED -----
<StringArray>
['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', nan]
Length: 6, dtype: str


In [7]:
# STEP 4: HANDLING MIXED BOOLEANS

print(df['active'].unique())

bool_map = {
    'Yes': True, 'Y': True, 1: True, '1': True,
    'No': False, 'N': False, 0: False, '0': False

}
# map the right values to what we want in the final version
df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)

print('----- FIX APPLIED -----')
print(df['active'].unique())

<StringArray>
['Y', '0', 'No', 'True', 'Yes', '1', 'False']
Length: 7, dtype: str
----- FIX APPLIED -----
[ True False]


In [15]:
# STEP 5: DATE PARSING

print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print('----- FIX APPLIED -----')
print(df['start_date'].dtype)

df = df.dropna(subset=['start_date', 'end_date'])


datetime64[us]
----- FIX APPLIED -----
datetime64[us]


In [16]:
# STEP 6: LOGICAL INTEGRITY (CLICKS vs IMPRESSIONS)

df = df.loc[:, ~df.columns.duplicated()] #located the columns that are duplicated and removes it

impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask, ['campaign_id', 'impressions', 'clicks']].head(3))

df = df.drop(df[impossible_mask].index)

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


In [17]:
# STEP 7: LOGICAL INTEGRITY (TIME TRAVEL)

# I wanna make sure we have kind of chronological order
# between start and end date and no instances
# where the end date is actually in the past compared to the start date

time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask, 'start_date'] + pd.Timedelta(days=30)

print('----- FIX APPLIED -----')
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

# Basically fix the right time


Empty DataFrame
Columns: [campaign_id, start_date, end_date]
Index: []
----- FIX APPLIED -----
Empty DataFrame
Columns: [campaign_id, start_date, end_date]
Index: []


In [18]:
# STEP 8: HANDLING OUTLIERS (WINSORIZING)

Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1
upper_limit = Q3 + (3 * IQR)

outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

# Basically, calculating the interquartile range which is kind of like an idea here
# I wanna see what is the usual range for the value spend.
# So what is considered like a normal variation for this value

# saying get the Q3, and add 3 times the usual range

print('----- FIX APPLIED -----')
df.loc[outlier_mask, 'spend'] = upper_limit
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))


     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375
----- FIX APPLIED -----
     campaign_id     spend
789    CMP-00790  8514.935
1443   CMP-01444  8514.935
1460   CMP-01461  8514.935


In [19]:
# STEP 9: STRING PARSING (FEATURE EXTRACTION)

print(df['campaign_name'].head(3))

df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')

print('----- FIX APPLIED -----')

print(df[['campaign_name', 'season']].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: str
----- FIX APPLIED -----
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


In [26]:
# STEP 10: CONVERSION MUST BE <= CLICKS

conv_error_mask = df['conversions'] > df['clicks']
# You can pick between Drop or set it values equal to Clicks
# But Drop it make it good quality
print(f"Rows with conversions > clicks: {conv_error_mask.sum()}")
if conv_error_mask.any():
    print(df.loc[conv_error_mask, ['campaign_id', 'clicks', 'conversions']].head(3))

# I decide to drop it
df = df.drop(df[conv_error_mask].index)

print('----- FIX APPLIED -----')
print(f"Rows with conversions > clicks: {(df['conversions'] > df['clicks']).sum()}")


Rows with conversions > clicks: 0
----- FIX APPLIED -----
Rows with conversions > clicks: 0


In [34]:
# STEP 11: HANDLING MISSING VALUES IN METRICS

# from Row 5: The columns "Conversions" is still NaN

# In marketing Analytics conversions is importants Metrics
# if empty it will make Sum and Mean wrong!

# Decide wether will add '0' or drop it

# Check empty values
print(f"NaNs in conversions: {df['conversions'].isna().sum()}")

# Add 0 and change type to 'int'
df['conversions'] = df['conversions'].fillna(0).astype(int)

print('----- FIX APPLIED -----')
print(f"NaNs in conversions: {df['conversions'].isna().sum()}")


NaNs in conversions: 0
----- FIX APPLIED -----
NaNs in conversions: 0


In [35]:
# STEP 12: Row-Level Duplicates

# We already removes duplicate colunms at STEP 6
# but never check Row Duplicates yet

# Check the dubplicates based on ID and start_date
duplicates_count = df.duplicated(subset=['campaign_id', 'start_date']).sum()
print(f"Duplicate rows found: {duplicates_count}")

# Just kill the dubplicates
df = df.drop_duplicates(subset=['campaign_id', 'start_date']) # or just check all columns

print('---- FIX APPLIED -----')
print(f"Duplicate rows remaining: {df.duplicated(subset=['campaign_id', 'start_date']).sum()}")

Duplicate rows found: 0
---- FIX APPLIED -----
Duplicate rows remaining: 0


In [ ]:
# Done

print(f"Final Cleaned Dataset: {df.shape[0]} rows")
print(df.info())
print()

audit_report = {
    "Total Rows Removed": 2020 - df.shape[0],
    "Unique Channels": df['channel'].nunique(),
    "Average Conversion Rate": (df['conversions'].sum() / df['clicks'].sum() * 100).round(2)
}

print("--- FINAL QUALITY AUDIT ---")
for key, value in audit_report.items():
    print(f"{key}: {value}")


Final Cleaned Dataset: 1668 rows
<class 'pandas.DataFrame'>
Index: 1668 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   campaign_id    1668 non-null   str           
 1   campaign_name  1668 non-null   str           
 2   start_date     1668 non-null   datetime64[us]
 3   end_date       1668 non-null   datetime64[us]
 4   channel        1581 non-null   str           
 5   impressions    1668 non-null   int64         
 6   clicks         1668 non-null   int64         
 7   spend          1668 non-null   float64       
 8   conversions    1668 non-null   int64         
 9   active         1668 non-null   bool          
 10  campaign_tag   1668 non-null   str           
 11  season         1668 non-null   str           
dtypes: bool(1), datetime64[us](2), float64(1), int64(3), str(5)
memory usage: 158.0 KB
None

--- FINAL QUALITY AUDIT ---
Total Rows Removed: 352
Unique Channels: